# AR Reconciliation API - Endpoint Testing

## Setup
```
uv sync
uv run alembic upgrade head
uv run python main.py   (in a separate terminal)
```

Base URL: `http://localhost:8000`

In [ ]:
import httpx
import time

BASE_URL = "http://localhost:8000"
client = httpx.Client(base_url=BASE_URL, timeout=30)
print("Client ready ✓")

## 1. Health Check
`GET /health` — Verify the server is running.

In [ ]:
resp = client.get("/health")
print(f"Status: {resp.status_code}")
resp.json()

## 2. Submit a Single Record
`POST /submit` — Submit one AR record for processing. Returns a workflow ID.

In [ ]:
record = {
    "customer_id": "CUST-0001",
    "customer_name": "Customer 0001",
    "customer_balance": 3486.58,
    "invoice_total": 3892.38,
    "invoice_applied_amount": 241.57,
    "invoice_exchange_rate": 1.1371,
    "payment_total": 3316.88,
    "payment_applied_amount": 2375.22,
    "payment_exchange_rate": 0.8873,
    "credit_total": 910.95,
    "credit_applied_amount": 676.97,
    "credit_exchange_rate": 1.0237,
    "adjustment_total": 1310.49,
    "adjustment_applied_amount": 944.51,
    "adjustment_exchange_rate": 1.1211,
}

resp = client.post("/submit", json=record)
print(f"Status: {resp.status_code}")
submit_result = resp.json()
workflow_id = submit_result["workflow_id"]
print(f"Workflow ID: {workflow_id}")
submit_result

## 3. Submit Duplicate (Idempotency Check)
Submitting the same `customer_id` again should return the existing workflow, not create a new one.

In [ ]:
# Submit the same record again — should be idempotent
resp = client.post("/submit", json=record)
print(f"Status: {resp.status_code}")
print("Expected: 'Duplicate submission' message")
resp.json()

## 4. Get Workflow Status
`GET /workflow/{id}` — Check the status and stage details of a workflow.

In [ ]:
# Wait a moment for background processing to complete
time.sleep(2)

resp = client.get(f"/workflow/{workflow_id}")
print(f"Status: {resp.status_code}")
workflow_detail = resp.json()
print(f"Workflow status: {workflow_detail['workflow']['status']}")
print(f"Current stage: {workflow_detail['workflow']['current_stage']}")
print(f"Number of stages recorded: {len(workflow_detail['stages'])}")
workflow_detail

## 5. Update a Record
`PUT /submit/{customer_id}` — Update an existing record and reprocess from scratch.

In [ ]:
# Update the record with a different balance
updated_record = record.copy()
updated_record["customer_balance"] = 5000.00
updated_record["invoice_total"] = 5500.00

resp = client.put("/submit/CUST-0001", json=updated_record)
print(f"Status: {resp.status_code}")
resp.json()

## 6. Bulk Upload CSV
`POST /bulk-upload` — Upload the CSV file to process multiple records at once.

In [ ]:
# Upload the CSV file for bulk processing
with open("../data/erp_export.csv", "rb") as f:
    resp = client.post(
        "/bulk-upload", files={"file": ("erp_export.csv", f, "text/csv")}
    )

print(f"Status: {resp.status_code}")
bulk_result = resp.json()
print(f"Total records: {bulk_result['total_records']}")
print(f"Submitted: {bulk_result['submitted']}")
print(f"Duplicates: {bulk_result['duplicates']}")
print(f"Workflow IDs (first 5): {bulk_result['workflow_ids'][:5]}")
bulk_result

## 7. List All Workflows
`GET /workflows` — List workflows with optional status filter and pagination.

In [ ]:
# Wait for background processing
time.sleep(3)

# List all workflows
resp = client.get("/workflows")
print(f"Status: {resp.status_code}")
workflows = resp.json()
print(f"Total workflows returned: {len(workflows)}")
# Show first 3
for w in workflows[:3]:
    print(
        f"  {w['id']} | {w['customer_id']} | {w['status']} | stage: {w['current_stage']}"
    )

In [ ]:
# Filter workflows by status
resp = client.get("/workflows", params={"status": "COMPLETED", "limit": 5})
print(f"Completed workflows: {resp.status_code}")
print(f"Count: {len(resp.json())}")

resp = client.get("/workflows", params={"status": "FAILED", "limit": 5})
print(f"\nFailed workflows: {resp.status_code}")
failed_workflows = resp.json()
print(f"Count: {len(failed_workflows)}")
for w in failed_workflows[:3]:
    print(f"  {w['id']} | {w['customer_id']} | retry_count: {w['retry_count']}")

## 8. Resume a Failed Workflow
`POST /resume/{id}` — Resume a workflow that failed (retries from the last successful stage).

In [ ]:
# Find a failed workflow to resume
resp = client.get("/workflows", params={"status": "FAILED", "limit": 1})
failed = resp.json()

if failed:
    failed_id = failed[0]["id"]
    print(f"Resuming workflow: {failed_id}")
    resp = client.post(f"/resume/{failed_id}")
    print(f"Status: {resp.status_code}")
    print(resp.json())

    # Check status after a moment
    time.sleep(2)
    resp = client.get(f"/workflow/{failed_id}")
    print(f"\nAfter resume - Status: {resp.json()['workflow']['status']}")
else:
    print("No failed workflows to resume (all succeeded!)")
    print(
        "Note: The system has a 20% random failure rate, so run bulk-upload again if needed"
    )

## 9. Dashboard Stats
`GET /stats` — Get aggregate statistics about all workflows.

In [ ]:
resp = client.get("/stats")
print(f"Status: {resp.status_code}")
stats = resp.json()
print(f"Total workflows: {stats['total_workflows']}")
print(f"Completed: {stats['completed']}")
print(f"Failed: {stats['failed']}")
print(f"Pending: {stats['pending']}")
print(f"Running: {stats['running']}")
print(f"Stale: {stats['stale']}")
print(f"Routing decisions: {stats['decisions']}")

## 10. Enhanced Workflow List
`GET /workflows/enhanced` — List workflows with staleness flag and last error message.

In [ ]:
# Enhanced list with error details
resp = client.get("/workflows/enhanced", params={"limit": 5})
print(f"Status: {resp.status_code}")
enhanced = resp.json()
for w in enhanced[:5]:
    stale_flag = " [STALE]" if w.get("is_stale") else ""
    error = f" | error: {w['last_error']}" if w.get("last_error") else ""
    print(f"  {w['customer_id']} | {w['status']}{stale_flag}{error}")

## 11. Export Results as CSV
`GET /export` — Download completed workflow results as a CSV file.

In [ ]:
# Export completed results as CSV
resp = client.get("/export")
print(f"Status: {resp.status_code}")
print(f"Content-Type: {resp.headers.get('content-type')}")
print("\nCSV Preview (first 500 chars):")
print(resp.text[:500])

## 12. Error Handling - Invalid Requests
Test that the API returns proper error responses for bad input.

In [ ]:
# Test 404 - workflow not found
resp = client.get("/workflow/nonexistent-id-12345")
print(f"GET /workflow/bad-id → {resp.status_code} (expected 404)")
print(f"  Response: {resp.json()}")

# Test 404 - resume non-existent workflow
resp = client.post("/resume/nonexistent-id-12345")
print(f"\nPOST /resume/bad-id → {resp.status_code} (expected 404)")
print(f"  Response: {resp.json()}")

# Test 400 - upload non-CSV file
resp = client.post(
    "/bulk-upload", files={"file": ("test.txt", b"not a csv", "text/plain")}
)
print(f"\nPOST /bulk-upload with .txt → {resp.status_code} (expected 400)")
print(f"  Response: {resp.json()}")

# Test 422 - missing required field
resp = client.post("/submit", json={})
print(f"\nPOST /submit with empty body → {resp.status_code} (expected 422)")
print(f"  Response: {resp.json()['detail'][0]['msg']}")

## 13. Resume All Failed Workflows
Loop through all failed workflows and attempt to resume them.

In [ ]:
# Resume all failed workflows
resp = client.get("/workflows", params={"status": "FAILED", "limit": 200})
failed = resp.json()
print(f"Found {len(failed)} failed workflows. Resuming...")

for w in failed:
    r = client.post(f"/resume/{w['id']}")
    print(f"  {w['customer_id']}: {r.json()['message']}")

# Wait and check final stats
time.sleep(5)
resp = client.get("/stats")
stats = resp.json()
print("\n--- Final Stats ---")
print(f"Completed: {stats['completed']}/{stats['total_workflows']}")
print(f"Still failed: {stats['failed']}")
print(f"Decisions: {stats['decisions']}")